In [0]:
%restart_python


In [0]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
#from data_processing import DataProcessing

In [0]:
# Execução
spark = SparkSession.builder.getOrCreate()
dataset_path = f"/Volumes/workspace/telco/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = spark.read.option("nullValue", " ").csv(dataset_path, header=True, multiLine=True, escape="'", inferSchema=True)
display(df)

In [0]:
print(df.schema)

In [0]:
num_features = [
    "tenure", 
    "MonthlyCharges", 
    "TotalCharges"]

cat_features = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod'
]

binary_features = ["SeniorCitizen", "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]

In [0]:
# from pyspark.sql.functions import col, count

# for feature in cat_features:
#     df.groupBy(feature).agg(count(col(feature)).alias("total")).display()

In [0]:
from data_utils import calculate_missing
spark = SparkSession.builder.getOrCreate()

missing_df = calculate_missing(df, spark)

In [0]:
#from pyspark.sql.functions import NoneType
per_thresh = 0.6  # Drop if column has more than 80% missing data

N = df.count()  # total count
to_drop_missing = [x.asDict()['Column'] for x in missing_df.select("Column").where(col("Number of Missing Values") / N >= per_thresh).collect()]

In [0]:
from pyspark.sql.types import BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, when, lower, trim

df_01 = df

for feature in binary_features:
    # Verifica o tipo da coluna no schema
    feature_type = df_01.schema[feature].dataType

    # Só faz cast para string se a coluna não for StringType
    if isinstance(feature_type, StringType):
        feature_clean = lower(trim(col(feature)))
    else:
        feature_clean = lower(trim(col(feature).cast("string")))

    df_01 = df_01.withColumn(
        feature,
        when(feature_clean.isin("yes", "1", "true"), 1)
        .when(feature_clean.isin("no", "0", "false"), 0)
        .otherwise(None)
        .cast(DoubleType())
    )

display(df_01.select(binary_features + num_features))

In [0]:
df_01.select(num_features).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "1%",
    "5%",
    "25%",
    "50%",
    "75%",
    "95%",
    "99%",
    "max"
).display()

In [0]:
from pyspark.sql.functions import col

feature = "TotalCharges"

q1, q3 = df_01.approxQuantile(feature, [0.25, 0.75], 0.01)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Feature:", feature)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Limite inferior:", lower_bound)
print("Limite superior:", upper_bound)

outliers_count = df_01.filter(
    (col(feature) < lower_bound) |
    (col(feature) > upper_bound)
).count()

total_count = df_01.count()

print("Quantidade de outliers:", outliers_count)
print("Percentual de outliers:", round(outliers_count / total_count * 100, 2), "%")

df_no_outliers = df_01.filter(
    (col(feature) >= lower_bound) &
    (col(feature) <= upper_bound)
)

print("Linhas antes:", total_count)
print("Linhas depois:", df_no_outliers.count())

display(df_no_outliers)

In [0]:
from pyspark.sql.functions import col, when, count, concat_ws, collect_list # isnan


def calculate_missing(input_df, show=True):
  """
  Helper function to calculate and display missing data
  """

  # First get count of missing values per column to get a singleton row DF
  missing_df_ = input_df.select([count(when(col(c).contains('None') | \
                                                  col(c).contains('NULL') | \
                                                  (col(c) == '' ) | \
                                                  col(c).isNull(), c)).alias(c) \
                                                  for c in input_df.columns
                                            ])

  # Transpose for better readability
  def TransposeDF(df, columns, pivotCol):
    """Helper function to transpose spark dataframe"""
    columnsValue = list(map(lambda x: str("'") + str(x) + str("',")  + str(x), columns))
    stackCols = ','.join(x for x in columnsValue)
    df_1 = df.selectExpr(pivotCol, "stack(" + str(len(columns)) + "," + stackCols + ")")\
            .select(pivotCol, "col0", "col1")
    final_df = df_1.groupBy(col("col0")).pivot(pivotCol).agg(concat_ws("", collect_list(col("col1"))))\
                  .withColumnRenamed("col0", pivotCol)
    return final_df

  missing_df_out_T = TransposeDF(
    spark.createDataFrame([{"Column":"Number of Missing Values"}]).join(missing_df_),
    missing_df_.columns,
    "Column").withColumn("Number of Missing Values", col("Number of Missing Values").cast("long"))

  if show:
    display(missing_df_out_T.orderBy("Number of Missing Values", ascending=False))

  return missing_df_out_T


calculate_missing(df_no_outliers)